In [ ]:
import numpy as np
from numpy.typing import NDArray
from typing import Sequence
type ArrayLike = NDArray | Sequence[int | float]

def ndgridm(N):
    """
    Expand index hypercube (MATLAB ndgridm equivalent)

    Parameters
    ----------
    N : array-like of ints
        Vector of max indices per dimension

    Returns
    -------
    NN : ndarray of shape (prod(N), len(N))
        Matrix of index combinations (1-based indexing)
    """
    N = np.asarray(N, dtype=int)

    # Base case
    if N.size == 1:
        return np.arange(1, N[0] + 1).reshape(-1, 1)

    # Recursive case
    n = np.arange(1, N[0] + 1)
    nn = ndgridm(N[1:])

    # Equivalent to MATLAB kron(n, ones(...))'
    first_col = np.kron(n, np.ones(np.prod(N[1:]), dtype=int)).reshape(-1, 1)

    # Equivalent to repmat(nn, [N(1) 1])
    rest = np.tile(nn, (N[0], 1))

    return np.hstack((first_col, rest))


def grid(m):
    temp = [np.arange(1, 1 + m[d]) for d in range(len(m))]
    S = np.meshgrid(*temp)
    S_arr = np.vstack([s.flatten() for s in S]).T
    return S_arr


In [ ]:
ndgridm([2,2,3])

In [ ]:
grid([2,2,3])

In [ ]:
def calc_eigenvalues(L: ArrayLike, m: int, d: int) -> NDArray:
    """
    Calculate eigenvalues of the Laplacian on [-L1,L1] x ... x [-Ld,Ld]
    with Dirichlet boundary conditions, returning the m smallest.

    Parameters
    ----------
    L : NDArray, shape (d,)
        Domain half-widths per dimension.
    m : int
        Number of eigenfunctions to return.
    d : int
        Number of input dimensions.

    Returns
    -------
    selected_per_dim_eigenvalues : NDArray, shape (m,d)
        The m smallest eigenvalues, sorted ascending.
    """
    L = np.asarray(L, dtype=float)
    
    # Number of indices per dimension, scaled by relative domain size
    N_per_dim = np.ceil(m ** (1 / d) * L / L.min()).astype(int)

    # Build full multi-index grid (Cartesian product of per-dim indices)
    temp = [np.arange(1, 1 + N_per_dim[dim]) for dim in range(d)]
    grids = np.meshgrid(*temp, indexing='ij')
    NN = np.vstack([g.ravel() for g in grids]).T      # (N_total, d)

    # Compute all eigenvalues
    per_dim_eigvals = np.square((np.pi * NN) / (2 * L)) # (N_total, d)
    all_eigenvalues = np.sum(per_dim_eigvals, axis=1)

    # Sort and keep the m smallest
    sort_idx = np.argsort(all_eigenvalues)[:m]
    selected_per_dim_eigenvalues = per_dim_eigvals[sort_idx]
    # NN = NN[sort_idx]

    return selected_per_dim_eigenvalues

def calc_eigenvectors(Xs: NDArray, L: ArrayLike, per_dim_eigvals: NDArray) -> NDArray:
    """Calculate eigenvectors of the Laplacian.
    These are used as basis vectors in the HSGP approximation.

    Parameters
    ----------
    Xs              : NDArray, shape (n_samples, d)
    L               : NDArray, shape (d,)
    per_dim_eigvals : NDArray, shape (m, d)
        Per dimension eigenvalues, with the smallest sum

    Returns
    -------
    phi : NDArray, shape (n_samples, m)
    """
    L = np.asarray(L, dtype=float)
    
    # (1, m, d) * (n_samples, 1, d) -> (n_samples, m, d)
    term1 = np.sqrt(per_dim_eigvals)[None, :, :]          # (1, m, d)
    term2 = Xs[:, None, :] + L[None, None, :]     # (n_samples, 1, d) + (1, 1, d)
    c = 1.0 / np.sqrt(L)                          # (d,)

    phi = c * np.sin(term1 * term2)               # (n_samples, m, d)
    return np.prod(phi, axis=-1)                  # (n_samples, m)

### Testing the eigenvalue function

In [ ]:
L = [1,1,0.5]
m = 50
n_dim = 3
margin = 1.8

per_dim_eigvals = calc_eigenvalues(np.array(L)*margin, m, n_dim)
print(np.sum(per_dim_eigvals, axis=1), len(per_dim_eigvals))


### Test eigenvector function

In [ ]:
# Create regular spaced grid grid as input vector
n_per_dim = 10
rng = np.random.default_rng(seed=0)

g = np.meshgrid(*[np.linspace(-l, l, n_per_dim) for l in L])
# Create the full grid and reshape to (n_samples, 3)
X = np.vstack([gi.ravel() for gi in g]).T  # (25, 3)
# X = 2*rng.random((125,3))
# X = rng.normal(0, (1,1,.1), size=(125,3))

eigvectors = calc_eigenvectors(X, L, per_dim_eigvals)
print(eigvectors)